# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
# imports

import os
import json
from typing import Dict, List, Tuple, Optional, Any, Union
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletion, ChatCompletionMessage, ChatCompletionToolMessageParam
from openai.types.chat.chat_completion_message_tool_call import ChatCompletionMessageToolCall
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message: str = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [4]:
os.getenv

<function os.getenv(key, default=None)>

In [5]:
# This function looks rather simpler than the one from my video, because we're taking advantage of the latest Gradio updates
# Note: This is the basic version without tools - see the enhanced version below

def chat_basic(message: str, history: List[Dict[str, str]]) -> str:
    messages: List[Dict[str, str]] = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response: ChatCompletion = openai.chat.completions.create(model=MODEL, messages=messages)
    print("choices caught -- ", response.choices)
    return response.choices[0].message.content or ""

# gr.ChatInterface(fn=chat_basic, type="messages").launch()

## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [ ]:
# Let's start by making a useful function

ticket_prices: Dict[str, str] = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city: str) -> str:
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [ ]:
get_ticket_price("Berlin")

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function: Dict[str, Any] = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools: List[Dict[str, Union[str, Dict[str, Any]]]] = [{"type": "function", "function": price_function}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [ ]:
def chat(message: str, history: List[Dict[str, str]]) -> str:
    messages: List[Dict[str, str]] = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response: ChatCompletion = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        tool_message: ChatCompletionMessage = response.choices[0].message
        tool_response, city = handle_tool_call(tool_message)
        
        # Convert ChatCompletionMessage to dict format for the messages list
        tool_message_dict = {
            "role": "assistant",
            "content": tool_message.content,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                } for tool_call in (tool_message.tool_calls or [])
            ]
        }
        
        messages.append(tool_message_dict)
        messages.append(tool_response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content or ""

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message: ChatCompletionMessage) -> Tuple[Dict[str, str], str]:
    print("message :", type(message), message)
    
    if not message.tool_calls:
        raise ValueError("No tool calls found in message")
    
    tool_call: ChatCompletionMessageToolCall = message.tool_calls[0]
    arguments: Dict[str, str] = json.loads(tool_call.function.arguments)
    city: str = arguments.get('destination_city', '')
    price: str = get_ticket_price(city)
    
    response: Dict[str, str] = {
        "role": "tool",
        "content": json.dumps({"destination_city": city, "price": price}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()